<a href="https://colab.research.google.com/github/purnimakushwaha/ITC101_Minor-project_python/blob/main/Weather_Intelligence_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# 🌦️ WEATHER INTELLIGENCE DASHBOARD
# Open-Meteo API + JSON + Pandas + Matplotlib + ipywidgets
# ============================================================

import requests
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from datetime import datetime
from IPython.display import display, HTML, clear_output


# ============================================================
# API URLs
# ============================================================

GEOCODING_API = "https://geocoding-api.open-meteo.com/v1/search"
WEATHER_API = "https://api.open-meteo.com/v1/forecast"


# ============================================================
# WEATHER CODE DESCRIPTION
# ============================================================

WEATHER_CODES = {
    0: ("☀️", "Clear Sky"),
    1: ("🌤️", "Mainly Clear"),
    2: ("⛅", "Partly Cloudy"),
    3: ("☁️", "Overcast"),
    45: ("🌫️", "Fog"),
    48: ("🌫️", "Rime Fog"),
    51: ("🌦️", "Light Drizzle"),
    53: ("🌦️", "Moderate Drizzle"),
    55: ("🌧️", "Dense Drizzle"),
    61: ("🌧️", "Light Rain"),
    63: ("🌧️", "Moderate Rain"),
    65: ("🌧️", "Heavy Rain"),
    71: ("🌨️", "Light Snow"),
    73: ("🌨️", "Moderate Snow"),
    75: ("❄️", "Heavy Snow"),
    80: ("🌦️", "Light Rain Showers"),
    81: ("🌧️", "Moderate Rain Showers"),
    82: ("⛈️", "Heavy Rain Showers"),
    95: ("⛈️", "Thunderstorm"),
    96: ("⛈️", "Thunderstorm + Hail"),
    99: ("⛈️", "Heavy Thunderstorm + Hail")
}


# ============================================================
# GLOBAL DATA
# ============================================================

weather_data = None
location_data = None
forecast_df = None


# ============================================================
# HEADER
# ============================================================

header = widgets.HTML(
    """
    <div style="
        background:#1976d2;
        color:white;
        padding:25px;
        border-radius:18px;
        text-align:center;
        font-family:Arial;
    ">

        <div style="
            font-size:32px;
            font-weight:bold;
        ">
            🌦️ WEATHER INTELLIGENCE DASHBOARD
        </div>

        <div style="
            margin-top:8px;
            font-size:15px;
        ">
            API-Based Python Minor Project
        </div>

    </div>
    """
)


# ============================================================
# CITY INPUT
# ============================================================

city_input = widgets.Text(
    placeholder="Enter city name e.g. Delhi",
    description="📍 City:",
    layout=widgets.Layout(
        width="70%"
    )
)


search_button = widgets.Button(
    description="🔍 Get Weather",
    button_style="primary"
)


# ============================================================
# OUTPUT AREA
# ============================================================

output = widgets.Output()


# ============================================================
# SEARCH CITY
# ============================================================

def search_city(city):

    params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json"
    }

    response = requests.get(
        GEOCODING_API,
        params=params,
        timeout=10
    )

    response.raise_for_status()

    data = response.json()

    if "results" not in data or not data["results"]:
        raise ValueError(
            "City not found. Please check the city name."
        )

    return data["results"][0]


# ============================================================
# GET WEATHER
# ============================================================

def get_weather(latitude, longitude):

    params = {

        "latitude": latitude,
        "longitude": longitude,

        "current": ",".join([
            "temperature_2m",
            "relative_humidity_2m",
            "apparent_temperature",
            "is_day",
            "precipitation",
            "weather_code",
            "wind_speed_10m",
            "wind_direction_10m"
        ]),

        "hourly": ",".join([
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation_probability",
            "precipitation",
            "wind_speed_10m"
        ]),

        "daily": ",".join([
            "weather_code",
            "temperature_2m_max",
            "temperature_2m_min",
            "sunrise",
            "sunset",
            "precipitation_sum",
            "precipitation_probability_max",
            "wind_speed_10m_max"
        ]),

        "timezone": "auto",

        "forecast_days": 7
    }

    response = requests.get(
        WEATHER_API,
        params=params,
        timeout=15
    )

    response.raise_for_status()

    return response.json()


# ============================================================
# WEATHER ANALYSIS
# ============================================================

def weather_advice(
    temperature,
    rain_probability,
    wind_speed,
    humidity
):

    advice = []


    if temperature >= 35:

        advice.append(
            "🥵 Very hot — stay hydrated."
        )

    elif temperature >= 30:

        advice.append(
            "☀️ Warm weather — keep water with you."
        )

    elif temperature <= 10:

        advice.append(
            "🧥 Cold weather — consider warm clothing."
        )

    else:

        advice.append(
            "🙂 Temperature is fairly comfortable."
        )


    if rain_probability >= 60:

        advice.append(
            "☔ High chance of rain — carry an umbrella."
        )

    elif rain_probability >= 30:

        advice.append(
            "🌦️ There may be some rain."
        )

    else:

        advice.append(
            "🌤️ Low chance of rain."
        )


    if wind_speed >= 30:

        advice.append(
            "💨 Strong winds — take care outdoors."
        )


    if humidity >= 80:

        advice.append(
            "💧 Humidity is high."
        )


    return advice


# ============================================================
# FETCH WEATHER BUTTON
# ============================================================

def fetch_weather(button=None):

    global weather_data
    global location_data
    global forecast_df


    city = city_input.value.strip()


    if not city:

        with output:

            clear_output()

            display(
                HTML(
                    """
                    <div style="
                        background:#fff3cd;
                        padding:18px;
                        border-radius:12px;
                    ">
                    ⚠️ Please enter a city name.
                    </div>
                    """
                )
            )

        return


    try:

        with output:

            clear_output()

            display(
                HTML(
                    """
                    <h3>
                    🔄 Fetching live weather data...
                    </h3>
                    """
                )
            )


        # ----------------------------------------------------
        # STEP 1: GEOCODING API
        # ----------------------------------------------------

        location_data = search_city(
            city
        )


        latitude = location_data["latitude"]
        longitude = location_data["longitude"]


        # ----------------------------------------------------
        # STEP 2: WEATHER API
        # ----------------------------------------------------

        weather_data = get_weather(
            latitude,
            longitude
        )


        # ----------------------------------------------------
        # STEP 3: CREATE DATAFRAME
        # ----------------------------------------------------

        daily = weather_data["daily"]


        forecast_df = pd.DataFrame({

            "Date": daily["time"],

            "Max Temperature (°C)":
                daily["temperature_2m_max"],

            "Min Temperature (°C)":
                daily["temperature_2m_min"],

            "Rain Probability (%)":
                daily["precipitation_probability_max"],

            "Rain (mm)":
                daily["precipitation_sum"],

            "Max Wind (km/h)":
                daily["wind_speed_10m_max"],

            "Sunrise":
                daily["sunrise"],

            "Sunset":
                daily["sunset"],

            "Weather Code":
                daily["weather_code"]
        })


        show_dashboard()


    except Exception as error:

        with output:

            clear_output()

            display(
                HTML(
                    f"""
                    <div style="
                        background:#ffebee;
                        padding:20px;
                        border-radius:12px;
                    ">

                    <h3>
                    ❌ Unable to fetch weather
                    </h3>

                    <p>
                    {str(error)}
                    </p>

                    <p>
                    Please check your internet
                    connection and city name.
                    </p>

                    </div>
                    """
                )
            )


# ============================================================
# DASHBOARD
# ============================================================

def show_dashboard():

    current = weather_data["current"]


    city = location_data.get(
        "name",
        "Unknown"
    )

    country = location_data.get(
        "country",
        ""
    )


    temperature = current[
        "temperature_2m"
    ]


    humidity = current[
        "relative_humidity_2m"
    ]


    apparent = current[
        "apparent_temperature"
    ]


    wind = current[
        "wind_speed_10m"
    ]


    precipitation = current[
        "precipitation"
    ]


    weather_code = current[
        "weather_code"
    ]


    emoji, condition = WEATHER_CODES.get(
        weather_code,
        ("🌍", "Unknown")
    )


    # --------------------------------------------------------
    # TODAY'S RAIN PROBABILITY
    # --------------------------------------------------------

    rain_probability = (
        forecast_df.iloc[0]
        ["Rain Probability (%)"]
    )


    # --------------------------------------------------------
    # ADVICE
    # --------------------------------------------------------

    advice = weather_advice(
        temperature,
        rain_probability,
        wind,
        humidity
    )


    with output:

        clear_output()


        # ----------------------------------------------------
        # LOCATION
        # ----------------------------------------------------

        display(
            HTML(
                f"""
                <div style="
                    text-align:center;
                    margin-top:20px;
                ">

                    <h1>
                    📍 {city}, {country}
                    </h1>

                    <p>
                    Latitude:
                    {location_data["latitude"]}
                    |
                    Longitude:
                    {location_data["longitude"]}
                    </p>

                </div>
                """
            )
        )


        # ----------------------------------------------------
        # CURRENT WEATHER CARD
        # ----------------------------------------------------

        display(
            HTML(
                f"""
                <div style="
                    background:#e3f2fd;
                    padding:25px;
                    border-radius:18px;
                    text-align:center;
                ">

                    <div style="
                        font-size:60px;
                    ">
                        {emoji}
                    </div>

                    <h1>
                        {temperature} °C
                    </h1>

                    <h2>
                        {condition}
                    </h2>

                    <p>
                        Feels like:
                        <b>{apparent} °C</b>
                    </p>

                </div>
                """
            )
        )


        # ----------------------------------------------------
        # WEATHER DETAILS
        # ----------------------------------------------------

        display(
            HTML(
                f"""
                <br>

                <table style="
                    width:100%;
                    border-collapse:collapse;
                    text-align:center;
                ">

                    <tr>

                        <td style="
                            padding:15px;
                            border:1px solid #ddd;
                        ">
                            💧<br>
                            <b>Humidity</b><br>
                            {humidity}%
                        </td>

                        <td style="
                            padding:15px;
                            border:1px solid #ddd;
                        ">
                            💨<br>
                            <b>Wind</b><br>
                            {wind} km/h
                        </td>

                        <td style="
                            padding:15px;
                            border:1px solid #ddd;
                        ">
                            🌧️<br>
                            <b>Rain</b><br>
                            {precipitation} mm
                        </td>

                        <td style="
                            padding:15px;
                            border:1px solid #ddd;
                        ">
                            ☔<br>
                            <b>Rain Chance</b><br>
                            {rain_probability}%
                        </td>

                    </tr>

                </table>
                """
            )
        )


        # ----------------------------------------------------
        # ADVICE
        # ----------------------------------------------------

        display(
            HTML(
                """
                <br>
                <h2>
                    💡 Weather Insights
                </h2>
                """
            )
        )


        for item in advice:

            display(
                HTML(
                    f"""
                    <div style="
                        background:#f5f5f5;
                        padding:12px;
                        margin:5px;
                        border-radius:10px;
                    ">
                    {item}
                    </div>
                    """
                )
            )


        # ----------------------------------------------------
        # 7-DAY FORECAST
        # ----------------------------------------------------

        display(
            HTML(
                """
                <br>
                <h2>
                    📅 7-Day Forecast
                </h2>
                """
            )
        )


        display(
            forecast_df[
                [
                    "Date",
                    "Max Temperature (°C)",
                    "Min Temperature (°C)",
                    "Rain Probability (%)",
                    "Rain (mm)",
                    "Max Wind (km/h)"
                ]
            ]
        )


        # ----------------------------------------------------
        # TEMPERATURE CHART
        # ----------------------------------------------------

        plt.figure(
            figsize=(10, 5)
        )


        plt.plot(
            forecast_df["Date"],
            forecast_df[
                "Max Temperature (°C)"
            ],
            marker="o",
            label="Maximum"
        )


        plt.plot(
            forecast_df["Date"],
            forecast_df[
                "Min Temperature (°C)"
            ],
            marker="o",
            label="Minimum"
        )


        plt.title(
            f"7-Day Temperature Forecast - {city}"
        )


        plt.xlabel(
            "Date"
        )


        plt.ylabel(
            "Temperature (°C)"
        )


        plt.xticks(
            rotation=45
        )


        plt.legend()


        plt.tight_layout()


        plt.show()


        # ----------------------------------------------------
        # RAIN PROBABILITY CHART
        # ----------------------------------------------------

        plt.figure(
            figsize=(10, 5)
        )


        plt.bar(
            forecast_df["Date"],
            forecast_df[
                "Rain Probability (%)"
            ]
        )


        plt.title(
            f"Rain Probability - {city}"
        )


        plt.xlabel(
            "Date"
        )


        plt.ylabel(
            "Probability (%)"
        )


        plt.xticks(
            rotation=45
        )


        plt.tight_layout()


        plt.show()


        # ----------------------------------------------------
        # SUNRISE / SUNSET
        # ----------------------------------------------------

        display(
            HTML(
                f"""
                <h2>
                    🌅 Sunrise & Sunset
                </h2>
                """
            )
        )


        sunrise = forecast_df.iloc[0][
            "Sunrise"
        ]


        sunset = forecast_df.iloc[0][
            "Sunset"
        ]


        display(
            HTML(
                f"""
                <div style="
                    display:flex;
                    justify-content:space-around;
                    background:#fff8e1;
                    padding:20px;
                    border-radius:15px;
                ">

                    <div>
                        🌅
                        <b>Sunrise</b><br>
                        {sunrise}
                    </div>

                    <div>
                        🌇
                        <b>Sunset</b><br>
                        {sunset}
                    </div>

                </div>
                """
            )
        )


# ============================================================
# BUTTON CONNECTION
# ============================================================

search_button.on_click(
    fetch_weather
)


# ============================================================
# DISPLAY APPLICATION
# ============================================================

display(header)


display(
    widgets.HBox(
        [
            city_input,
            search_button
        ],
        layout=widgets.Layout(
            justify_content="center",
            margin="20px 0"
        )
    )
)


display(output)


print(
    "🌦️ Weather Intelligence Dashboard "
    "is ready!"
)

HTML(value='\n    <div style="\n        background:#1976d2;\n        color:white;\n        padding:25px;\n    …

Output()

🌦️ Weather Intelligence Dashboard is ready!
